In [1]:
###############################################################################
# Author: Edward E. Daisey
# Class: Modeling & Simulation of Complex Systems
# Professor: Professor Wiley
# Date: 10th of February, 2026
# Title: Assignment 3 Problem 6
###############################################################################

################################### Results ##################################
# What this script does:
# (1) Samples random 2x2 matrices A = [[a,b],[c,d]] with a,b,c,d ~ U[-1,1]
# independently (continuous uniform).
# (2) Computes tau = tr(A) = a + d and delta = det(A) = ad - bc.
# (3) Computes discriminant D = tau^2 - 4*delta.
# (4) Classifies the fixed point at the origin using the tau-delta chart:
# - saddles: delta < 0
# - nodes: delta > 0 and D > 0
#   * stable nodes: tau < 0
#   * unstable nodes: tau > 0
# - spirals: delta > 0 and D < 0
#   * stable spirals: tau < 0
#   * unstable spirals: tau > 0
# - degenerate/star nodes: delta > 0 and D = 0 (measure-zero)
# - centers: delta > 0 and D < 0 and tau = 0 (measure-zero)
# - non-isolated points: delta = 0 (measure-zero)
#
# Notes on "measure-zero" categories:
# With continuous sampling, exact equalities (delta=0, D=0, tau=0) occur with
# probability 0; numerically we detect near zero via eps.
#
# Reproducibility:
# (0) Copy this code to choice IDE.
# (1) Execute code.
# (2) The numerical frequencies will be printed to the terminal.
#
# References:
# [0] Makoto Matsumoto and Takuji Nishimura. 1998. Mersenne twister:
# a 623-dimensionally equidistributed uniform pseudo-random number generator.
# ACM Trans. Model. Comput. Simul. 8, 1 (Jan. 1998), 3–30.
# https://doi.org/10.1145/272991.272995
###############################################################################

def Abs(x):
    return -x if x < 0.0 else x

def NearlyZero(x, eps):
    return Abs(x) <= eps

class MT19937:
    # MT19937 (Mersenne Twister) PRNG, 32-bit output. Old, but I know it well
    # and feel comfortable implementing this PRNG.

    def __init__(self, seed):
        self.w = 32
        self.n = 624
        self.m = 397
        self.r = 31
        self.a = 0x9908B0DF
        self.u = 11
        self.d = 0xFFFFFFFF
        self.s = 7
        self.b = 0x9D2C5680
        self.t = 15
        self.c = 0xEFC60000
        self.l = 18
        self.f = 1812433253
        self.lowerMask = (1 << self.r) - 1
        self.upperMask = ((1 << self.w) - 1) ^ self.lowerMask
        self.state = [0] * self.n
        self.index = self.n
        self.Seed(seed)

    def Seed(self, seed):
        # Initializes internal state from an integer seed.
        self.index = self.n
        self.state[0] = seed & 0xFFFFFFFF
        for i in range(1, self.n):
            prev = self.state[i - 1]
            self.state[i] = (
                self.f * (prev ^ (prev >> (self.w - 2))) + i
            ) & 0xFFFFFFFF

    def Twist(self):
        # Generates the next n values from the current state.
        for i in range(self.n):
            x = (self.state[i] & self.upperMask) + (
                self.state[(i + 1) % self.n] & self.lowerMask
            )
            xA = x >> 1
            if (x & 1) != 0:
                xA ^= self.a
            self.state[i] = self.state[(i + self.m) % self.n] ^ xA
        self.index = 0

    def NextUInt32(self):
        # Returns the next 32-bit unsigned integer.
        if self.index >= self.n:
            self.Twist()

        y = self.state[self.index]
        self.index += 1

        # Tempering.
        y ^= ((y >> self.u) & self.d)
        y ^= ((y << self.s) & self.b) & 0xFFFFFFFF
        y ^= ((y << self.t) & self.c) & 0xFFFFFFFF
        y ^= (y >> self.l)
        return y & 0xFFFFFFFF

    def UniformMinus1To1(self):
        # Returns a float uniformly distributed on [-1, 1).
        u01 = self.NextUInt32() / 4294967296.0  # 2^32
        return 2.0 * u01 - 1.0

def ClassifyOne(a, b, c, d, eps):
    # Classifies the origin fixed point type for A = [[a,b],[c,d]].
    tau = a + d
    delta = a * d - b * c
    disc = tau * tau - 4.0 * delta  # D

    deltaZero = NearlyZero(delta, eps)
    tauZero = NearlyZero(tau, eps)
    discZero = NearlyZero(disc, eps)

    if deltaZero:
        return "nonIsolatedFixedPoints"

    if tauZero and (delta > eps) and (disc < -eps):
        return "centers"

    if (delta > eps) and discZero:
        return "degenerateStarNodes"

    if delta < -eps:
        return "saddles"

    if (delta > eps) and (disc > eps):
        if tau < -eps:
            return "stableNodes"
        if tau > eps:
            return "unstableNodes"
        return "other"

    if (delta > eps) and (disc < -eps):
        if tau < -eps:
            return "stableSpirals"
        if tau > eps:
            return "unstableSpirals"
        return "other"
    return "other"

def RunSimulation(numTrials, seed, eps):
    # Runs the Monte Carlo simulation and returns (counts, freqs).
    rng = MT19937(seed)

    keys = [
        "centers",
        "degenerateStarNodes",
        "nonIsolatedFixedPoints",
        "stableNodes",
        "unstableNodes",
        "saddles",
        "stableSpirals",
        "unstableSpirals",
        "other",
    ]

    counts = {k: 0 for k in keys}

    for _ in range(numTrials):
        a = rng.UniformMinus1To1()
        b = rng.UniformMinus1To1()
        c = rng.UniformMinus1To1()
        d = rng.UniformMinus1To1()

        label = ClassifyOne(a, b, c, d, eps)
        counts[label] += 1

    freqs = {k: counts[k] / float(numTrials) for k in keys}
    return counts, freqs

def PrintReport(numTrials, seed, eps, counts, freqs):
    # Prints an aligned results report.
    print("Monte Carlo: Random 2x2 Matrices With Entries U[-1, 1]")
    print("------------------------------------------------------")
    print("Trials:", numTrials)
    print("Seed: ", seed)
    print("Eps: ", eps)
    print()

    print(f"{'Type':24s} {'Count':>10s} {'Frequency':>12s}")
    print(f"{'-'*24} {'-'*10} {'-'*12}")

    order = [
        "saddles",
        "stableSpirals",
        "unstableSpirals",
        "stableNodes",
        "unstableNodes",
        "centers",
        "degenerateStarNodes",
        "nonIsolatedFixedPoints",
        "other",
    ]

    for key in order:
        print(f"{key:24s} {counts[key]:10d} {freqs[key]:12.6f}")

def Main():
    numTrials = 200000  # >= 100000
    seed = 10000  # Based on my experience, a good initial seed.
    eps = 1e-12

    counts, freqs = RunSimulation(numTrials, seed, eps)
    PrintReport(numTrials, seed, eps, counts, freqs)

if __name__ == "__main__":
    Main()

Monte Carlo: Random 2x2 Matrices With Entries U[-1, 1]
------------------------------------------------------
Trials: 200000
Seed:  10000
Eps:  1e-12

Type                          Count    Frequency
------------------------ ---------- ------------
saddles                       99810     0.499050
stableSpirals                 32106     0.160530
unstableSpirals               32015     0.160075
stableNodes                   18151     0.090755
unstableNodes                 17918     0.089590
centers                           0     0.000000
degenerateStarNodes               0     0.000000
nonIsolatedFixedPoints            0     0.000000
other                             0     0.000000
